# Paper 3 Gate 1 Scale-Up Runner

This Colab notebook runs the first real public-benchmark scale-up for the Gate 1 question:

- MSC valid: **32 conversations**
- LongMemEval-S cleaned: **12 conversations**
- model: `qwen25_15b`
- budgets: `0.20, 0.35, 0.50`

It will:

1. clone the repo,
2. install dependencies,
3. download and normalize MSC valid and LongMemEval-S cleaned,
4. run oracle headroom + refinement studies for both benchmarks,
5. print the main report paths,
6. optionally zip the outputs for download or Drive backup.

Recommended runtime:

- **A100** or **L4** preferred
- **T4** is possible, but slower
- runtime type: **GPU**


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/SteveMama/rt-geometry-memory.git"
REPO_DIR = "/content/rt-geometry-memory"

MODEL = "qwen25_15b"
BUDGETS = "0.20,0.35,0.50"
TARGET_STRIDE = 4
MAX_TARGET_TURNS = 16
LONGMEM_MAX_TURNS = 40
RUN_PREFIX = "paper3_gate1_scaleup"

# Optional persistence
SAVE_TO_DRIVE = False
DRIVE_ROOT = "/content/drive/MyDrive/rt_gate1_scaleup"

print({
    "MODEL": MODEL,
    "BUDGETS": BUDGETS,
    "TARGET_STRIDE": TARGET_STRIDE,
    "MAX_TARGET_TURNS": MAX_TARGET_TURNS,
    "LONGMEM_MAX_TURNS": LONGMEM_MAX_TURNS,
    "RUN_PREFIX": RUN_PREFIX,
    "SAVE_TO_DRIVE": SAVE_TO_DRIVE,
})


In [ ]:
# ── GPU check ────────────────────────────────────────────────────────────────
import os
import subprocess
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. In Colab, switch to Runtime > Change runtime type > GPU.")

subprocess.run(["nvidia-smi"], check=False)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print("Drive root:", DRIVE_ROOT)


In [ ]:
# ── Clone and install ───────────────────────────────────────────────────────
import os
%cd /content
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!bash scripts/colab_setup.sh


In [ ]:
# ── Optional: Hugging Face login (only if model download fails) ─────────────
# import os
# os.environ["HF_TOKEN"] = "hf_..."
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])


In [ ]:
# ── Download and normalize benchmarks ───────────────────────────────────────
import os
%cd {REPO_DIR}

MSC_RAW = f"{REPO_DIR}/benchmarks/msc_valid_raw.jsonl"
MSC_NORM = f"{REPO_DIR}/benchmarks/msc_valid_normalized.jsonl"
LONGMEM_RAW = f"{REPO_DIR}/benchmarks/longmemeval_s_cleaned_raw.json"
LONGMEM_NORM = f"{REPO_DIR}/benchmarks/longmemeval_s_cleaned_normalized.jsonl"

if not os.path.exists(MSC_NORM):
    !python scripts/download_public_benchmark.py --benchmark msc_valid --output "{MSC_RAW}"
    !python scripts/prepare_public_benchmark_jsonl.py --format msc --input "{MSC_RAW}" --output "{MSC_NORM}" --family msc_valid
else:
    print("MSC valid already present:", MSC_NORM)

if not os.path.exists(LONGMEM_NORM):
    !python scripts/download_public_benchmark.py --benchmark longmemeval_s_cleaned --output "{LONGMEM_RAW}"
    !python scripts/prepare_public_benchmark_jsonl.py --format longmemeval --input "{LONGMEM_RAW}" --output "{LONGMEM_NORM}" --family longmemeval_s_cleaned
else:
    print("LongMemEval-S cleaned already present:", LONGMEM_NORM)

for label, path in [("MSC valid", MSC_NORM), ("LongMemEval-S cleaned", LONGMEM_NORM)]:
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0.0
    line_count = sum(1 for _ in open(path, "r", encoding="utf-8")) if os.path.exists(path) else 0
    print(f"{label}: {line_count} conversations, {size_mb:.1f} MB -> {path}")


In [ ]:
# ── Run the 32-conversation MSC + 12-conversation LongMemEval scale-up ─────
%cd {REPO_DIR}
MSC_PATH = f"{REPO_DIR}/benchmarks/msc_valid_normalized.jsonl"
LONGMEM_PATH = f"{REPO_DIR}/benchmarks/longmemeval_s_cleaned_normalized.jsonl"

!bash scripts/run_paper3_gate1_scaleup.sh \
    "{MSC_PATH}" \
    "{LONGMEM_PATH}" \
    "{MODEL}" \
    "{BUDGETS}" \
    {TARGET_STRIDE} \
    {MAX_TARGET_TURNS} \
    {LONGMEM_MAX_TURNS} \
    "{RUN_PREFIX}"


In [ ]:
# ── Summarize the key outputs ────────────────────────────────────────────────
from pathlib import Path

root = Path(REPO_DIR)
paths = [
    root / f"results/paper3/harm_oracle/{RUN_PREFIX}_oracle_msc_valid_32conv/report.md",
    root / f"results/paper3/studies/{RUN_PREFIX}_refinement_msc_valid_32conv/study_report.md",
    root / f"results/paper3/studies/{RUN_PREFIX}_refinement_msc_valid_32conv/pairwise_report.md",
    root / f"results/paper3/harm_oracle/{RUN_PREFIX}_oracle_longmemeval_s_cleaned_12conv/report.md",
    root / f"results/paper3/studies/{RUN_PREFIX}_refinement_longmemeval_s_cleaned_12conv/study_report.md",
    root / f"results/paper3/studies/{RUN_PREFIX}_refinement_longmemeval_s_cleaned_12conv/pairwise_report.md",
]

for path in paths:
    print("\n===", path, "===")
    if path.exists():
        text = path.read_text(encoding="utf-8")
        print(text[:4000])
    else:
        print("missing")


In [ ]:
# ── Optional: zip outputs for download / backup ─────────────────────────────
%cd {REPO_DIR}
ZIP_PATH = f"{REPO_DIR}/{RUN_PREFIX}_results.zip"
!rm -f "{ZIP_PATH}"
!zip -r "{ZIP_PATH}" \
    "results/paper3/harm_oracle/{RUN_PREFIX}_oracle_msc_valid_32conv" \
    "results/paper3/studies/{RUN_PREFIX}_refinement_msc_valid_32conv" \
    "results/paper3/harm_oracle/{RUN_PREFIX}_oracle_longmemeval_s_cleaned_12conv" \
    "results/paper3/studies/{RUN_PREFIX}_refinement_longmemeval_s_cleaned_12conv"
print("Wrote:", ZIP_PATH)

if SAVE_TO_DRIVE:
    !cp "{ZIP_PATH}" "{DRIVE_ROOT}/"
    print("Copied zip to Drive:", DRIVE_ROOT)
